# NLP Forward Example

Define the NLP data, training, and projection configurations in the notebook, then build labels and train KKT-HardNet directly through the `kkthn` package.

In [ ]:
from pathlib import Path
import json
import time

from kkthn.problems import build_problem_bundle, write_json
from kkthn.projection import ProjectionSettings
from kkthn.training import KKTTrainConfig, train_kkt_hardnet


def make_run_dir(name):
    root = Path.cwd().resolve()
    notebook_dir = root if root.name == "notebooks" else root / "notebooks"
    run_dir = notebook_dir / "_runs" / name / time.strftime("%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def make_train_config(config, proj):
    return KKTTrainConfig(
        epochs=int(config["epochs"]),
        batch_size=int(config["batch_size"]),
        learning_rate=float(config["learning_rate"]),
        train_frac=float(config["train_frac"]),
        hidden_size=int(config["hidden_size"]),
        hidden_layers=int(config["hidden_layers"]),
        seed=int(config["seed"]),
        dtype=str(config["dtype"]),
        print_every=int(config["print_every"]),
        projection=ProjectionSettings(**proj),
    )


In [ ]:
DATA = {
    "type": "nlp",
    "n_x": 2,
    "n_y": 4,
    "n_eq": 1,
    "n_ineq": 1,
    "N_samples": 12,
    "N_points": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "solver": "SCS",
    "is_diag_Q": False,
    "q_diag_shift": 0.5,
    "nl_margin": 1.0,
    "bound_margin": 1.0,
    "bound_scale": 0.2,
    "param_scale": 0.4,
    "force_regenerate": True,
    "x_L": [-1.0],
    "x_U": [1.0],
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


In [ ]:
run_dir = make_run_dir("nlp_forward")
bundle = build_problem_bundle("nlp", Path("."), DATA)

write_json(run_dir / "data.json", DATA)
write_json(run_dir / "config.json", CONFIG)
write_json(run_dir / "proj.json", PROJ)
write_json(run_dir / "label_metadata.json", bundle.metadata)

print(f"X shape: {bundle.X.shape}")
print(f"Y shape: {bundle.Y.shape}")
print(json.dumps(bundle.metadata, indent=2)[:2000])

result = train_kkt_hardnet(
    model=bundle.model,
    X=bundle.X,
    Y=bundle.Y,
    cfg=make_train_config(CONFIG, PROJ),
    output_dir=run_dir,
    metadata=bundle.metadata,
)


In [ ]:
summary = {
    "output_dir": result["output_dir"],
    "dims": result["dims"],
    "final": result["final"],
    "component_time_percent": result["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))
